# **INTRO**

As an initial step in the project, we conducted a focused exploratory data analysis of the LEAF prompt dataset in order to understand its structure, quality, and semantic characteristics before making any embedding-related design choices. The dataset contains 20,000 records and 20 fields, with no missing values, which makes it suitable for a robust semantic retrieval pipeline. Our analysis examined the distribution of languages, categories, subcategories, difficulty levels, text lengths, placeholders, and tags, as well as the distinction between semantic fields and metadata. This preliminary stage was essential to ensure that the embedding strategy would be grounded in the actual properties of the corpus rather than selected arbitrarily.

In [ ]:
import json
import pandas as pd
import numpy as np
from collections import Counter
import re

INPUT_FILE = r"LEAF-promptkaban-dataset\dataset.json"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total records:", len(data))
print("Data structure type:", type(data))
print("First record keys:", list(data[0].keys()))

Total records: 20000
Data structure type: <class 'list'>
First record keys: ['id', 'author_reputation', 'version', 'fork_count', 'likes', 'upvotes', 'downvotes', 'views', 'uses', 'created_at', 'title', 'content', 'category', 'subcategory', 'tags', 'has_placeholders', 'placeholders', 'difficulty', 'language', 'target_model']


In [ ]:
df = pd.DataFrame(data)
print("Dataset shape:", df.shape)
df.head(3)
print("DATA TYPES:")
print(df.dtypes)
print()

print("MISSING VALUES:")
print(df.isna().sum().sort_values(ascending=False))

Dataset shape: (20000, 20)
DATA TYPES:
id                   object
author_reputation     int64
version               int64
fork_count            int64
likes                 int64
upvotes               int64
downvotes             int64
views                 int64
uses                  int64
created_at           object
title                object
content              object
category             object
subcategory          object
tags                 object
has_placeholders       bool
placeholders         object
difficulty           object
language             object
target_model         object
dtype: object

MISSING VALUES:
id                   0
author_reputation    0
version              0
fork_count           0
likes                0
upvotes              0
downvotes            0
views                0
uses                 0
created_at           0
title                0
content              0
category             0
subcategory          0
tags                 0
has_placeholders     0
pl

**Dataset overview**

The dataset contains **20,000 prompt records** and **20 fields**, providing a sufficiently large corpus for building and testing a semantic retrieval pipeline. The absence of missing values across all columns is particularly valuable, as it removes the need for imputation or record filtering before the embedding stage. The schema also shows a clear separation between textual, categorical, and behavioral metadata, which supports a modular design for downstream retrieval and rerankin

In [ ]:
print("LANGUAGE DISTRIBUTION:")
print(df["language"].value_counts())
print()

print("TOP 20 CATEGORIES:")
print(df["category"].value_counts().head(20))
print()

print("TOP 20 SUBCATEGORIES:")
print(df["subcategory"].value_counts().head(20))
print()

print("DIFFICULTY DISTRIBUTION:")
print(df["difficulty"].value_counts())
print()

print("HAS_PLACEHOLDERS DISTRIBUTION:")
print(df["has_placeholders"].value_counts())

LANGUAGE DISTRIBUTION:
language
en    19991
es        5
fr        2
no        1
zh        1
Name: count, dtype: int64

TOP 20 CATEGORIES:
category
creative-writing    1556
marketing           1257
sales               1186
legal                990
customer-support     899
design               891
education            622
data-analysis        596
translation          531
finance              428
coding               404
game-design          390
cybersecurity        390
research             383
devops               365
science-research     345
localization         342
productivity         333
gaming               273
recruiting           257
Name: count, dtype: int64

TOP 20 SUBCATEGORIES:
subcategory
ticket-responses            235
contract-drafting           214
cold-outreach               206
character-development       194
plot-structure              192
ui-ux                       183
follow-ups                  176
fiction                     167
dialogue                    147
obje

**Core distributions for embeddings**

**Language distribution**

The corpus is overwhelmingly **English-dominant** *(19,991 out of 20,000 records)*, with only a negligible number of prompts in other languages. This strongly supports the use of an English-focused embedding model, while suggesting that multilingual handling can be treated as an edge case rather than a primary design requirement.

**Category distribution**

The category distribution reveals a broad thematic spread, with strong concentration in domains such as creative writing, marketing, sales, legal, and customer support. This variety is important because the embedding space must capture meaning across multiple domains rather than within a single narrow topic. At the same time, the presence of dominant categories may bias retrieval quality if evaluation queries are not selected carefully across the corpus.

**Subcategory distribution**

Subcategories appear more fine-grained and operationally informative than top-level categories. Labels such as ticket-responses, contract-drafting, cold-outreach, and plot-structure suggest that the dataset is not only topically diverse but also organized around task intent, which may be beneficial for semantic disambiguation when prompts are short or underspecified.

**Difficulty distribution**

The dataset is centered around intermediate-level prompts, followed by beginner, advanced, and expert prompts. This indicates that the corpus mostly represents practical, applied requests rather than highly specialized expert-only prompts. For embeddings, this matters because semantic variation is likely driven more by task formulation and domain vocabulary than by technical depth alone.

**Placeholder distribution**

Most prompts **do not contain placeholders**, while a meaningful minority does. This suggests that placeholders are not universal enough to define the corpus, but they are common enough to deserve explicit consideration during preprocessing. The embedding pipeline should therefore remain robust to both fully specified prompts and reusable template-style prompt

In [ ]:
df["title_len_chars"] = df["title"].fillna("").astype(str).str.len()
df["content_len_chars"] = df["content"].fillna("").astype(str).str.len()

df["title_len_words"] = df["title"].fillna("").astype(str).str.split().str.len()
df["content_len_words"] = df["content"].fillna("").astype(str).str.split().str.len()

print("TEXT LENGTH STATS:")
print(df[[
    "title_len_chars", "content_len_chars",
    "title_len_words", "content_len_words"
]].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

TEXT LENGTH STATS:
       title_len_chars  content_len_chars  title_len_words  content_len_words
count     20000.000000       20000.000000     20000.000000       20000.000000
mean         29.839050         338.263600         4.389250          54.703300
std          12.095531         250.593884         2.200381          46.813192
min           1.000000           4.000000         1.000000           1.000000
25%          23.000000         127.000000         3.000000          24.000000
50%          30.000000         323.000000         4.000000          45.000000
75%          36.000000         452.000000         5.000000          67.000000
90%          43.000000         681.100000         7.000000         128.000000
95%          49.000000         854.000000         9.000000         162.000000
99%          65.000000        1091.000000        12.000000         207.000000
max         200.000000        1441.000000        43.000000         272.000000


**Text length analysis**

The length statistics show a clear asymmetry between the two main textual fields: ***titles are generally short***, while **contents are** substantially **longer** and more informative. On average, **titles contain roughly 4.4 word**s, whereas **contents contain about 54.7 words**, with a long upper tail extending beyond 200 words. This confirms that content is the main carrier of semantic information, while title functions more as a compressed cue than a complete description. Extremely short prompts and very long prompts coexist in the same corpus, which may create heterogeneity in embedding quality and should be considered during model selection and evaluation

In [ ]:
def is_very_short(text, threshold=4):
    return len(str(text).split()) <= threshold

def has_all_caps(text):
    s = str(text)
    letters = [c for c in s if c.isalpha()]
    if not letters:
        return False
    upper_ratio = sum(c.isupper() for c in letters) / len(letters)
    return upper_ratio > 0.7

def has_informal_style(text):
    s = str(text).lower()
    patterns = [
        r"\bidk\b", r"\bwats\b", r"\bpls\b", r"\bwut\b", r"\bim\b",
        r"\bcuz\b", r"\btho\b", r"\bgimme\b", r"\babt\b", r"\blol\b"
    ]
    return any(re.search(p, s) for p in patterns)

df["title_very_short"] = df["title"].apply(is_very_short)
df["title_all_caps_style"] = df["title"].apply(has_all_caps)
df["title_informal_style"] = df["title"].apply(has_informal_style)

print("TITLE NOISE CHECK:")
print("Very short titles (%):", round(df["title_very_short"].mean() * 100, 2))
print("All-caps style titles (%):", round(df["title_all_caps_style"].mean() * 100, 2))
print("Informal-style titles (%):", round(df["title_informal_style"].mean() * 100, 2))

TITLE NOISE CHECK:
Very short titles (%): 64.33
All-caps style titles (%): 4.61
Informal-style titles (%): 2.09


**Title quality / noise check**

The title analysis indicates that a large share of titles are very short (over 64%), while all-caps and strongly informal forms are present but much less frequent. This pattern suggests that titles cannot be assumed to be consistently descriptive: in many cases they provide useful context, but in others they are vague, colloquial, or stylistically noisy. As a result, titles may enrich the embedding input, but they should not be treated as a reliable standalone semantic field.

In [ ]:
df["n_placeholders"] = df["placeholders"].apply(lambda x: len(x) if isinstance(x, list) else 0)

print("PLACEHOLDER STATS:")
print(df["n_placeholders"].describe())
print()

all_placeholders = Counter()
for row in df["placeholders"]:
    if isinstance(row, list):
        all_placeholders.update(row)

print("TOP 30 PLACEHOLDERS:")
for ph, cnt in all_placeholders.most_common(30):
    print(f"{ph}: {cnt}")

PLACEHOLDER STATS:
count    20000.000000
mean         0.236650
std          0.708076
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         10.000000
Name: n_placeholders, dtype: float64

TOP 30 PLACEHOLDERS:
industry: 248
company_name: 215
topic: 213
product_name: 174
audience: 161
tone: 141
number: 90
genre: 90
language: 81
subject: 59
grade_level: 52
jurisdiction: 50
code_snippet: 38
format: 35
department: 35
duration: 35
context: 34
target_language: 31
product_type: 31
customer_name: 29
text_input: 27
job_title: 25
research_topic: 24
setting: 24
style: 22
project_name: 22
project_type: 21
platform: 21
role: 21
target_audience: 20


**Placeholder analysis**

The placeholder statistics show that the median prompt contains no placeholders, but the maximum reaches 10, confirming that some prompts are highly templated. The most frequent placeholders — such as industry, company_name, topic, product_name, and audience — capture domain-specific variables rather than arbitrary syntax. This means placeholders may preserve useful structural intent, especially in reusable business and professional prompts. Their treatment should therefore be cautious: removing them entirely could discard part of the prompt’s functional meaning.

In [ ]:
df["n_tags"] = df["tags"].apply(lambda x: len(x) if isinstance(x, list) else 0)

print("TAG COUNT STATS:")
print(df["n_tags"].describe())
print()

all_tags = Counter()
for row in df["tags"]:
    if isinstance(row, list):
        all_tags.update(row)

print("TOP 50 TAGS:")
for tag, cnt in all_tags.most_common(50):
    print(f"{tag}: {cnt}")

TAG COUNT STATS:
count    20000.000000
mean         4.153300
std          0.846307
min          0.000000
25%          3.000000
50%          4.000000
75%          5.000000
max          7.000000
Name: n_tags, dtype: float64

TOP 50 TAGS:
template: 469
beginner: 409
compliance: 352
writing: 332
documentation: 328
feedback: 323
analysis: 309
email: 293
python: 253
small-business: 239
dialogue: 237
automation: 232
marketing: 228
branding: 228
communication: 223
basics: 219
checklist: 218
gdpr: 217
strategy: 214
hiring: 213
help: 207
planning: 204
saas: 190
best-practices: 187
onboarding: 186
statistics: 186
architecture: 182
worldbuilding: 181
e-commerce: 179
performance: 176
fantasy: 171
metrics: 169
methodology: 169
script: 167
copywriting: 164
questions: 162
sql: 162
follow-up: 162
design: 161
pricing: 159
engagement: 159
templates: 153
comparison: 153
review: 149
structure: 147
productivity: 147
negotiation: 146
startup: 144
workflow: 136
optimization: 133


**Tag analysis**

Tags are relatively dense and well populated, with a median of **4 tags per prompt** and an upper bound of 7. The most frequent tags indicate recurring themes such as **templates, compliance, documentation, analysis, email, Python, and marketing**. This suggests that tags provide concise semantic descriptors that may complement the main text, particularly when the prompt itself is brief or underspecified. At the same time, tags are editorial metadata, so their contribution should be validated empirically rather than assumed to improve retrieval by default.

In [ ]:
print("5 SHORTEST PROMPTS BY WORD COUNT:")
shortest = df.sort_values("content_len_words").head(5)
for _, row in shortest.iterrows():
    print("\nID:", row["id"])
    print("Title:", row["title"])
    print("Content:", row["content"])
    print("Category:", row["category"], "| Subcategory:", row["subcategory"])


print("\n 5 LONGEST PROMPTS BY WORD COUNT:")
longest = df.sort_values("content_len_words", ascending=False).head(5)
for _, row in longest.iterrows():
    print("\nID:", row["id"])
    print("Title:", row["title"])
    print("Content preview:", row["content"][:1200])
    print("Category:", row["category"], "| Subcategory:", row["subcategory"])


5 SHORTEST PROMPTS BY WORD COUNT:

ID: pk_16772
Title: help pls
Content: translate
Category: translation | Subcategory: general-translation

ID: pk_16680
Title: help
Content: help
Category: translation | Subcategory: minimal-request

ID: pk_06172
Title: legal question
Content: regulations
Category: legal | Subcategory: regulatory-analysis

ID: pk_07754
Title: microservices help
Content: microservices???
Category: architecture | Subcategory: design-patterns

ID: pk_06995
Title: transcreation
Content: transcreation
Category: translation | Subcategory: transcreation

 5 LONGEST PROMPTS BY WORD COUNT:

ID: pk_16332
Title: company wants to rebrand and i got put in charge
Content preview: so basically my company is doing this whole rebrand and somehow i ended up being the project lead even though i have no formal design training i just have good taste apparently which is not the same as knowing how to execute a rebrand so im kinda panicking because theres like a whole process people go throu

**Shortest prompts by word count**

The shortest prompts illustrate a critical challenge for the embedding stage: some records contain extremely limited lexical information, such as one-word or near one-word requests. In these cases, semantic meaning is only partially expressed in the content field and is more likely to emerge from the combination of title, category, subcategory, and tags. These examples show why an embedding strategy based on content alone may underperform on a non-trivial subset of the datase

**Longest prompts by word count**

The longest prompts resemble full user problem descriptions rather than compact prompt templates. They contain narrative context, uncertainty, and implicit goals, which provide rich semantic material for dense representations. These cases are likely to benefit strongly from modern sentence-level or text-level embeddings, since the main challenge is not lexical sparsity but preserving topical and functional coherence across a longer sequence

In [ ]:
print("5 PROMPTS WITH PLACEHOLDERS:")
with_ph = df[df["has_placeholders"] == True].head(5)

for _, row in with_ph.iterrows():
    print("\nID:", row["id"])
    print("Title:", row["title"])
    print("Placeholders:", row["placeholders"])
    print("Content:", row["content"][:1000])


5 PROMPTS WITH PLACEHOLDERS:

ID: pk_00898
Title: Create Employee Handbook Section
Placeholders: ['policy_topic']
Content: Write a professional employee handbook section about {{policy_topic}}. Include an introduction explaining the policy's purpose, specific guidelines employees should follow, examples of acceptable and unacceptable practices, and information about who to contact with questions. Use a warm but professional tone suitable for new hires.

ID: pk_05166
Title: Vendor Contract Review
Placeholders: ['vendor_contract_text']
Content: Review this vendor contract and identify potential risks and unfavorable clauses. Focus on: indemnification provisions, limitation of liability caps, auto-renewal terms, data handling and security obligations, termination convenience clauses, and governing law implications. Provide specific red flags in a numbered list with recommended revisions for each issue identified.

{{vendor_contract_text}}

ID: pk_02982
Title: Structure lab protocol steps


In [ ]:
semantic_fields = [
    "title", "content", "category", "subcategory", "tags", "placeholders", "language"
]

metadata_fields = [
    "author_reputation", "version", "fork_count", "likes", "upvotes", "downvotes",
    "views", "uses", "created_at", "has_placeholders", "difficulty", "target_model"
]

print("SEMANTIC FIELDS:")
for c in semantic_fields:
    print("-", c)

print("\n METADATA / RANKING / FILTERING FIELDS:")
for c in metadata_fields:
    print("-", c)

SEMANTIC FIELDS:
- title
- content
- category
- subcategory
- tags
- placeholders
- language

 METADATA / RANKING / FILTERING FIELDS:
- author_reputation
- version
- fork_count
- likes
- upvotes
- downvotes
- views
- uses
- created_at
- has_placeholders
- difficulty
- target_model


**Semantic vs metadata fields**

The distinction between semantic fields and metadata fields is well justified by the dataset design. Fields such as content, title, category, subcategory, tags, and placeholders contribute directly to meaning or task intent, whereas engagement-related and behavioral variables such as likes, views, uses, and reputation are better interpreted as ranking or filtering signals. This separation supports a clean architecture in which semantic similarity is modeled independently from popularity or platform performance

**Final Overwiev**


The exploratory analysis highlighted several findings that are directly relevant to the embedding component. First, the corpus is overwhelmingly English-dominant, which supports the adoption of an English-oriented embedding model. Second, the content field emerges as the main semantic carrier, both by dataset definition and by observed text-length patterns, whereas title appears too short and inconsistent to be used as a standalone semantic representation. At the same time, categories, subcategories, and tags may provide useful contextual support, especially for short or underspecified prompts. Finally, engagement-related variables such as likes, upvotes, views, uses, and author reputation should not be incorporated into the semantic embedding itself, since they reflect behavioral or ranking signals rather than prompt meaning. These observations provide a principled basis for designing the embedding input and for keeping semantic representation separate from later ranking and reranking stages

# **PARTE NUOVA**
# **Definition and Construction of Embedding Input Configurations**

Based on the exploratory data analysis, we designed a single enriched textual representation for embedding generation. Since the `content` field contains the actual instruction or task expressed by each prompt, it was treated as the primary semantic component. At the same time, the analysis showed that additional fields such as `category`, `subcategory`, `tags`, and `title` can provide meaningful context about the prompt domain, topic, and intended usage.

To preserve this information while keeping the input compatible with sentence embedding models, each semi-structured prompt record was converted into a natural-language description. This avoids representing the data as a raw sequence of fields and instead provides the model with a coherent textual input.

**The final embedding input** was constructed according to the following template:

***"The task is: {content}***

***This prompt belongs to the {category} category and the {subcategory} subcategory.***

***It is related to {tags}.***

***The prompt title is "{title}"***

This structure gives priority to the prompt content while integrating the other semantic fields as contextual signals. The objective is to obtain a representation that remains faithful to the original prompt, but is richer and more informative than the content alone, enabling the embedding model to encode both the task and its semantic context.

In [ ]:
import pandas as pd
import re

def clean_text(text):
    if text is None:
        return ""
    if isinstance(text, float) and pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
def format_tags(tags, max_tags=4):
    cleaned_tags = [clean_text(tag) for tag in tags if clean_text(tag)]
    cleaned_tags = cleaned_tags[:max_tags]

    if len(cleaned_tags) == 0:
        return ""
    elif len(cleaned_tags) == 1:
        return cleaned_tags[0]
    else:
        return ", ".join(cleaned_tags[:-1]) + f", and {cleaned_tags[-1]}"

In [ ]:
for col in ["content", "category", "subcategory", "title"]:
    print(col, (df[col].astype(str).str.strip() == "").sum())

print("empty tag lists:", df["tags"].apply(lambda x: len(x) == 0).sum())

content 0
category 0
subcategory 0
title 0
empty tag lists: 6


In [ ]:
def build_embedding_text(row):
    tags = format_tags(row["tags"], max_tags=4)

    sentences = [
        f"The task is: {clean_text(row['content'])}.",
        f"This prompt belongs to the {clean_text(row['category'])} category and the {clean_text(row['subcategory'])} subcategory."
    ]

    if tags:
        sentences.append(f"It is related to {tags}.")

    sentences.append(f'The prompt title is "{clean_text(row["title"])}".')

    return " ".join(sentences)


df["embedding_text"] = df.apply(build_embedding_text, axis=1)

The template was then implemented by creating a new `embedding_text` column. Before doing so, we applied only minimal preprocessing to ensure that the textual fields were consistently formatted.

The main semantic fields (`content`, `category`, `subcategory`, and `title`) were checked and found to be non-empty for all records, while six prompts had no associated tags. Therefore, the tag-related sentence was included only when tags were available.

As a result, each row of the dataset is mapped to a single natural-language input string. This representation preserves the original prompt content while adding the available semantic context in a controlled and consistent way, and it will be used as the shared input for the embedding models evaluated later.

some example:

In [ ]:
for i in [0, 10, 25, 100, 250]:
    print("ID:", df.loc[i, "id"])
    print(df.loc[i, "embedding_text"])

ID: pk_03138
The task is: Create a comprehensive brand identity brief document for a new sustainable fashion startup. Include all essential sections: Executive Summary, Brand Purpose and Mission, Target Audience Persona (demographics, psychographics, behaviors), Brand Positioning statement, Brand Personality (with detailed trait descriptions), Competitive Analysis summary, Core Brand Values, Brand Promise, Tone of Voice guidelines (with examples for different scenarios), and Visual Direction overview. Format this as a professional document a design agency would deliver to stakeholders.. This prompt belongs to the brand-identity category and the strategic-brief subcategory. It is related to brand-brief, sustainable-fashion, strategy, and identity. The prompt title is "Complete Brand Identity Brief".
ID: pk_15071
The task is: my sales pipleine has too many stages whats the best setup. This prompt belongs to the sales category and the crm-setup subcategory. It is related to pipeline, stag

Before generating the embeddings, we checked the length of the final `embedding_text` representation. This step was necessary because the enriched input combines several fields of the original dataset and although each field is individually manageable, their combination could potentially create input texts that are too long for some embedding models to process fully.

This is important because embedding models have a maximum input length. If a text exceeds this limit, the model may truncate part of the input, meaning that some semantic information could be ignored during embedding generation. For this reason, we analyzed the word length distribution of the enriched representation before moving to the embedding stage.

The objective of this check was to verify whether the enriched inputs remained compact enough and to identify whether additional truncation strategies would be needed.

In [ ]:
df["embedding_text_len_words"] = df["embedding_text"].str.split().str.len()

print("ENRICHED INPUT LENGTH STATS")
print(df[["embedding_text_len_words"]].describe())

print("\n WORD LENGTH PERCENTILES")
print(df["embedding_text_len_words"].quantile([0.90, 0.95, 0.99]))

ENRICHED INPUT LENGTH STATS
       embedding_text_len_words
count              20000.000000
mean                  85.937650
std                   48.458988
min                   25.000000
25%                   55.000000
50%                   76.000000
75%                   99.000000
max                  310.000000

 WORD LENGTH PERCENTILES
0.90    162.0
0.95    196.0
0.99    242.0
Name: embedding_text_len_words, dtype: float64


# **Token Length Check with the Baseline Embedding Model**

After checking the word length of the enriched representation, we also evaluated its token length using the tokenizers of the selected embedding models. This step is important because embedding models process tokenized text and have a fixed maximum sequence length. Therefore, even if the enriched inputs appear reasonably short in terms of words, they may still exceed the model-specific token limit.

We selected three embedding models according to the final goal of the project: semantic retrieval over a corpus of prompts. `all-MiniLM-L6-v2` was used as a lightweight and efficient baseline, `BAAI/bge-base-en-v1.5` as a strong retrieval-oriented model with a good quality-to-size ratio, and `intfloat/e5-base-v2` as another retrieval-oriented model specifically designed for query-passage matching.

Since E5 models are trained to distinguish between queries and passages, we used the prefix `passage:` for the prompt representations in the corpus. User queries will later be encoded with the prefix `query:`. For MiniLM and BGE, we used the standard enriched representation without additional prefixes.

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

model_name = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Model max sequence length:", model.max_seq_length)

df["embedding_text_tokens_minilm"] = df["embedding_text"].apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=True))
)

print("TOKEN LENGTH STATS - MiniLM")
print(df[["embedding_text_tokens_minilm"]].describe())

above_limit = (df["embedding_text_tokens_minilm"] > model.max_seq_length).sum()
percentage_above_limit = above_limit / len(df) * 100

print("\nInputs above model limit:", above_limit)
print("Percentage above model limit:", percentage_above_limit)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model max sequence length: 256
TOKEN LENGTH STATS - MiniLM
       embedding_text_tokens_minilm
count                  20000.000000
mean                     122.805900
std                       55.425537
min                       37.000000
25%                       84.000000
50%                      117.000000
75%                      148.000000
max                      391.000000

Inputs above model limit: 591
Percentage above model limit: 2.955


In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

df["embedding_text_e5"] = "passage: " + df["embedding_text"]


def check_token_length(model_name, text_column):
    model = SentenceTransformer(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    safe_name = model_name.split("/")[-1].replace("-", "_").replace(".", "_")
    token_col = f"tokens_{safe_name}"

    df[token_col] = df[text_column].apply(
        lambda x: len(tokenizer.encode(x, add_special_tokens=True))
    )

    above_limit = (df[token_col] > model.max_seq_length).sum()
    percentage_above_limit = above_limit / len(df) * 100

    print("=" * 80)
    print("Model:", model_name)
    print("Model max sequence length:", model.max_seq_length)
    print(df[[token_col]].describe())
    print("\nInputs above model limit:", above_limit)
    print("Percentage above model limit:", percentage_above_limit)

    return token_col

check_token_length(
    "intfloat/e5-base-v2",
    text_column="embedding_text_e5"
)

check_token_length(
    "BAAI/bge-base-en-v1.5",
    text_column="embedding_text"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: intfloat/e5-base-v2
Model max sequence length: 512
       tokens_e5_base_v2
count       20000.000000
mean          124.805900
std            55.425537
min            39.000000
25%            86.000000
50%           119.000000
75%           150.000000
max           393.000000

Inputs above model limit: 0
Percentage above model limit: 0.0


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: BAAI/bge-base-en-v1.5
Model max sequence length: 512
       tokens_bge_base_en_v1_5
count             20000.000000
mean                122.805900
std                  55.425537
min                  37.000000
25%                  84.000000
50%                 117.000000
75%                 148.000000
max                 391.000000

Inputs above model limit: 0
Percentage above model limit: 0.0


'tokens_bge_base_en_v1_5'

The token length check showed that the enriched representation is fully or almost fully compatible with the selected models. The MiniLM baseline has a maximum sequence length of 256 tokens and would truncate 591 inputs, corresponding to 2.955% of the dataset. This makes MiniLM useful as a lightweight baseline, but less suitable as the final embedding model for the enriched representation.

In contrast, BGE and E5 provide better compatibility with the enriched input. BGE was selected as a strong retrieval-oriented alternative from the proposed open-source models, while E5 was selected because its query-passage training format is well aligned with the final semantic search task. E5, with a maximum sequence length of 512 tokens, can process all enriched inputs without truncation. For this reason, the final embedding comparison focuses on BGE and E5, while MiniLM remains a baseline reference.

**Embedding Generation with the Selected Models**

Based on the token length analysis, we selected BGE and E5 for the final embedding generation. BGE was chosen as a strong retrieval-oriented model with a good quality-to-size ratio, while E5 was selected as another retrieval-oriented model designed for query-passage matching.

Both models are applied to the same enriched prompt representation, so that the comparison focuses on the embedding model rather than on different input configurations. For E5, the `passage:` prefix is used for corpus embeddings, consistently with the model input format.

In [ ]:
bge_model_name = "BAAI/bge-base-en-v1.5"

bge_model = SentenceTransformer(bge_model_name)

embeddings_bge = bge_model.encode(
    df["embedding_text"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("BGE embeddings shape:", embeddings_bge.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

BGE embeddings shape: (20000, 768)


In [ ]:


df["embedding_text_e5"] = "passage: " + df["embedding_text"]

e5_model_name = "intfloat/e5-base-v2"

e5_model = SentenceTransformer(e5_model_name)

embeddings_e5 = e5_model.encode(
    df["embedding_text_e5"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("E5 embeddings shape:", embeddings_e5.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

E5 embeddings shape: (20000, 768)


In [ ]:
np.save("embeddings_bge_enriched.npy", embeddings_bge)
np.save("embeddings_e5_enriched.npy", embeddings_e5)

retrieval_metadata_cols = [
    "id",
    "likes",
    "upvotes",
    "title",
    "content",
    "category",
    "subcategory",
    "tags",
    "difficulty",
    "language",
    "target_model",
    "embedding_text"
]

df[retrieval_metadata_cols].to_csv("retrieval_metadata_enriched.csv", index=False)

# **CREATING THE VECTOR DATABASE**
Now that we have embedded the dataset into vectors, we can proceed to build the vectorial database for a fast and efficient retrieval of our contents.
We have decided to build 2 databases: one with only config A and one with only config B.

We do so as to be able to play around with all possible combinations of data to find the best approach when we will do operations like retrieval through cosine similarity, or retrieval based on metadata.

We will use ChromaDB as our vector database.

In [ ]:
!pip install chromadb

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\claud\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import numpy as np
import pandas as pd
import chromadb

config_a_bge = np.load("embeddings_bge_enriched.npy")
config_b_e5 = np.load("embeddings_e5_enriched.npy")
metadata = pd.read_csv("retrieval_metadata_enriched.csv")
print(f"Config A: {len(config_a_bge)}")
print(f"Config B: {len(config_b_e5)}")
print(f"CSV rows: {len(metadata)}")

Config A: 20000
Config B: 20000
CSV rows: 20000


In [ ]:
chromaDB = chromadb.PersistentClient(path="/content")

In [ ]:
# We create the two (empty) databases

config_a_bge_collection = chromaDB.get_or_create_collection(name="config_a_bge")
config_b_e5_collection = chromaDB.get_or_create_collection(name="config_b_e5")

**Filling the databases**

Once the two empty databases have been created, we go on to the next part: actually loading the vectors we prepared earlier to the databases. We will create a function that takes care of the vector storage, and we will implement, together with those, metadata which will help us in the later parts (retrieval).

In [ ]:
def database_adder(database, vectors, metadata):
  configuration = vectors.tolist()
  for batch_start in range(0, len(configuration), 5000):
    batch_end = min(batch_start+5000, len(configuration))
    ids = []
    embeddings = []
    documents = []
    metadatas = []
    for vector in range(batch_start, batch_end):
      ids.append(str(metadata["id"][vector]))
      embeddings.append(configuration[vector])
      documents.append(str(metadata["content"][vector]))
      metadatas.append({
          "title":str(metadata["title"][vector]),
          "category":str(metadata["category"][vector]),
          "subcategory":str(metadata["subcategory"][vector]),
          "tags":str(metadata["tags"][vector]),
          "difficulty":str(metadata["difficulty"][vector]),
          "likes":int(metadata["likes"][vector]),
          "upvotes":int(metadata["upvotes"][vector]),
      })

    database.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
  )

In [ ]:
# WE LOAD THE VECTORS INTO THE DATABASES

database_adder(config_a_bge_collection, config_a_bge, metadata)
database_adder(config_b_e5_collection, config_b_e5, metadata)

**Sanity checks**

Now that we have built the databases, we can run some quick sanity checks to verify that they are up to standard.

In [ ]:
# Verify that vectors can be retrieved
config_a_bge_collection.get(ids="pk_03138", include=["embeddings"])

{'ids': ['pk_03138'],
 'embeddings': array([[ 1.17667783e-02, -1.46160582e-02, -4.34236042e-02,
          4.82978486e-02,  5.96906208e-02,  3.09621077e-02,
          1.25752874e-02,  2.47084890e-02, -1.42644746e-02,
         -2.18808539e-02, -7.05975443e-02, -3.72187700e-03,
         -4.14964445e-02, -5.87511361e-02, -3.29741910e-02,
          3.15574706e-02,  3.83483879e-02,  4.95844185e-02,
         -2.16067657e-02,  5.25023136e-03, -2.79747695e-02,
          1.49746416e-02,  4.27214913e-02,  4.12761047e-02,
          4.91511710e-02,  2.46165711e-02,  4.90815379e-02,
          5.53898606e-03, -5.97133934e-02,  3.23177092e-02,
          8.23126547e-03, -5.78515716e-02,  1.36808138e-02,
         -1.56985205e-02,  4.56831604e-02, -2.84981392e-02,
         -1.08437939e-02,  2.70252675e-02, -8.06096662e-03,
         -3.81667688e-02, -5.47065139e-02,  1.36140222e-02,
          1.64998844e-02, -2.54272148e-02,  3.60108796e-03,
          2.34830473e-02, -2.94698104e-02,  1.62578076e-02,
    

In [ ]:
config_b_e5_collection.get(ids="pk_03138")

{'ids': ['pk_03138'],
 'embeddings': None,
 'documents': ['Create a comprehensive brand identity brief document for a new sustainable fashion startup. Include all essential sections: Executive Summary, Brand Purpose and Mission, Target Audience Persona (demographics, psychographics, behaviors), Brand Positioning statement, Brand Personality (with detailed trait descriptions), Competitive Analysis summary, Core Brand Values, Brand Promise, Tone of Voice guidelines (with examples for different scenarios), and Visual Direction overview. Format this as a professional document a design agency would deliver to stakeholders.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'title': 'Complete Brand Identity Brief',
   'subcategory': 'strategic-brief',
   'difficulty': 'expert',
   'tags': "['brand-brief', 'sustainable-fashion', 'strategy', 'identity']",
   'category': 'brand-identity',
   'upvotes': 277,
   'likes': 478}]}

In [ ]:
# Check the length of the databases. Should be equal to the embedded vectors
print("Config A count:", config_a_bge_collection.count())
print("Config B count:", config_b_e5_collection.count())

Config A count: 20000
Config B count: 20000


# STEP 3: Cosine Similarity Retrieval

**Recap and explanation of this step:**
After having completed step 1 and 2 by embedding all 20,000 prompts into vectors and storing them in a ChromaDB vector database, we are ready to handle the query part.

This section handles the query side of the user, and given a natural language search query we will:
1. Embed the query using the **same model** that was used to build each collection — **BGE** for Config A and **E5** for Config B. The same-model rule (query and corpus must share the embedding space) still holds within each collection; the two collections are two independent retrieval systems running in parallel.
2. Search ChromaDB using **cosine similarity** to find the closest matching prompts.
3. Return ranked results (that will later be passed to the reranker in Step 4).

We run this on **both Config A** (BGE embeddings of the enriched representation) and **Config B** (E5 embeddings of the enriched representation, with the `query:` prefix applied on the query side) so the reranker team can compare both.

## 1. Install dependencies

In [ ]:
!pip install -q sentence-transformers chromadb


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\claud\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2. Load the embedding models

We load **both** embedding models used in Step 2: BGE for Config A and E5 for Config B.

The "same model on both sides" rule still applies, but **per collection**: each query is encoded with the model that built the collection it's being searched against. Inside a single search, query and corpus must share the embedding space, otherwise cosine similarity is meaningless. Across the two collections, the two retrieval systems run independently in parallel.

In [ ]:
from sentence_transformers import SentenceTransformer


bge_model = SentenceTransformer("BAAI/bge-base-en-v1.5")
e5_model = SentenceTransformer("intfloat/e5-base-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 3. Connect to ChromaDB

Step 2 created two collections in ChromaDB (`config_a_bge`, `config_b_e5`). We connect to the database here.

In [ ]:
import chromadb

# ChromaDB
chromaDB = chromadb.PersistentClient(path="/content")
config_a_chromadb = chromaDB.get_collection(name="config_a_bge")
config_b_chromadb = chromaDB.get_collection(name="config_b_e5")

print(f"ChromaDB Config A Loading Check: {config_a_chromadb.count()} prompts")
print(f"ChromaDB Config B Loading Check: {config_b_chromadb.count()} prompts")

ChromaDB Config A Loading Check: 20000 prompts
ChromaDB Config B Loading Check: 20000 prompts


## 4. Retrieval function

We implement a retrieval function for ChromaDB that:
- Takes a natural language query and converts it into a vector
- Searches the vector database using cosine similarity
- Returns results sorted from most to least relevant, each with a similarity score and full metadata (including `likes`, `upvotes`, `difficulty` for the reranker)

In [ ]:
import numpy as np
import time

def retrieve_chromadb(query: str, collection, embedding_model, top_k: int = 20, query_prefix=""):
    query_vector = embedding_model.encode(
        query_prefix + query,
        convert_to_numpy=True
    ).tolist()

    # Searching ChromaDB
    start = time.time()
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    latency = time.time() - start

    # ChromaDB returns lists-of-lists (one per query), so we take index [0]
    ids       = results["ids"][0]
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    # ChromaDB distance = 1 - cosine_similarity, so we convert back
    formatted = []
    for rank, (pid, doc, meta, dist) in enumerate(zip(ids, documents, metadatas, distances), start=1):
        formatted.append({
            "rank":rank,
            "id":pid,
            "similarity":round(1 - dist, 4),
            "distance":round(dist, 4),
            "title":meta.get("title", ""),
            "category":meta.get("category", ""),
            "subcategory":meta.get("subcategory", ""),
            "tags":meta.get("tags", ""),
            "difficulty":meta.get("difficulty", ""),
            "likes":meta.get("likes", 0),
            "upvotes":meta.get("upvotes", 0),
            "content":doc,
        })

    return formatted, latency


def print_results(results, query, label):
    print(f"\n{'='*70}")
    print(f"Query: \"{query}\"")
    print(f"{'='*70}")
    for r in results:
        print(f"\n  #{r['rank']}  [{r['id']}]  similarity={r['similarity']}")
        print(f"  Title      : {r['title']}")
        print(f"  Category   : {r['category']} > {r['subcategory']}")
        print(f"  Tags       : {r['tags']}")
        print(f"  Difficulty : {r['difficulty']} | Likes: {r['likes']} | Upvotes: {r['upvotes']}")
        print(f"  Content    : {r['content']}...")
        print(f"  {'-'*66}")

## 5. Running a search

Change `QUERY` to whatever you want to search for.

In [ ]:
QUERY = "help me write a professional email to a client"
TOP_K = 20

results_a, _ = retrieve_chromadb(QUERY, config_a_chromadb, bge_model, top_k=TOP_K)
results_b, _ = retrieve_chromadb(QUERY, config_b_chromadb, e5_model, top_k=TOP_K, query_prefix="query: ")

print_results(results_a[:5], QUERY, "Config A (BGE)")
print_results(results_b[:5], QUERY, "Config B (E5)")


Query: "help me write a professional email to a client"

  #1  [pk_00021]  similarity=0.5297
  Title      : A small request if you don't mind - new to sales and feeling a bit lost
  Category   : sales > cold-outreach
  Tags       : ['cold email', 'templates', 'beginner', 'office supplies']
  Difficulty : beginner | Likes: 3 | Upvotes: 0
  Content    : Good day to you! I hope this message finds you well and that you are having a lovely week so far. My name is Gerald and I recently started a new job in sales for a small company that sells office supplies. I must admit I am not very familiar with how to write good cold emails to potential customers and I was wondering if you might be so kind as to help me with some templates or examples? I would really appreciate any guidance you could offer. Please forgive me if my request is too basic or if I am not explaining myself clearly. Thank you so much in advance for your time and patience. Warmest regards, Gerald...
  -------------------------

## 6. A/B comparison across multiple queries

We test several queries and compare whether Config A or Config B returns better top results.
This is the A/B comparison the challenge evaluation requires.

In [ ]:
test_queries = [
    "write Python code to scrape a website",
    "summarize a legal contract",
    "create a marketing post for Instagram",
    "explain machine learning to a beginner",
    "generate a cold outreach email for sales",
]

print("Running A/B comparison across test queries...\n")

for query in test_queries:
    res_a, _ = retrieve_chromadb(query, config_a_chromadb, bge_model, top_k=5)
    res_b, _ = retrieve_chromadb(query, config_b_chromadb, e5_model, top_k=5, query_prefix="query: ")

    ids_a = [r["id"] for r in res_a]
    ids_b = [r["id"] for r in res_b]
    overlap = len(set(ids_a) & set(ids_b))

    print(f"\nQUERY: \"{query}\"")
    for i in range(5):
        match = "✓" if res_a[i]["id"] == res_b[i]["id"] else "✗"
        print(f"  #{i+1} {match}  A: [{res_a[i]['id']}] sim={res_a[i]['similarity']} | {res_a[i]['title'][:40]}")
        print(f"       B: [{res_b[i]['id']}] sim={res_b[i]['similarity']} | {res_b[i]['title'][:40]}")
    print(f"  → Shared prompts in top 5: {overlap}/5")

Running A/B comparison across test queries...


QUERY: "write Python code to scrape a website"
  #1 ✗  A: [pk_07903] sim=0.2662 | pyhton scrip for data viz
       B: [pk_05798] sim=0.6721 | Clean and Merge Multiple Datasets
  #2 ✗  A: [pk_12201] sim=0.254 | pyhton script for data
       B: [pk_19610] sim=0.6696 | pyhton script for netwerk monitering
  #3 ✗  A: [pk_10393] sim=0.2502 | code plz
       B: [pk_05582] sim=0.6653 | need a script for my youtube ads
  #4 ✗  A: [pk_05798] sim=0.2478 | Clean and Merge Multiple Datasets
       B: [pk_18916] sim=0.6646 | pyhton scrip for data
  #5 ✗  A: [pk_02369] sim=0.2403 | Data Cleaning Code Template
       B: [pk_07903] sim=0.6605 | pyhton scrip for data viz
  → Shared prompts in top 5: 2/5

QUERY: "summarize a legal contract"
  #1 ✓  A: [pk_17110] sim=0.6719 | Contract Summary Generator
       B: [pk_17110] sim=0.7917 | Contract Summary Generator
  #2 ✓  A: [pk_04414] sim=0.5863 | Contract Summary Extraction
       B: [pk_04414] sim=0.7659 |

## 7. Evaluate retrieval quality — Precision@K and MRR

To measure quality we picked a small set of queries where we manually mark which results are truly relevant.

- **Precision@K**: of the top K results, what fraction are actually relevant?
- **MRR (Mean Reciprocal Rank)**: on average, how high up does the first relevant result appear? (1.0 = always #1, 0.5 = first relevant is at #2, etc.)

In [ ]:
GROUND_TRUTH = {
    "create a social media marketing campaign": [
        "pk_02364",  # Social media visual campaign
        "pk_00017",  # Social Media Content Calendar
        "pk_14568",  # Social Media Content Calendar Generator
        "pk_16478",  # Viral Social Media Post Formulas
        "pk_01267",  # Create Hashtag Strategy Plan
    ],
    "write a response to a negative customer review": [
        "pk_06217",  # Negative Review Response Templates (ticket-responses)
        "pk_04336",  # Negative Review Response Templates (ticket-responses)
        "pk_19170",  # Negative Review Response Templates (marketing)
        "pk_11972",  # Create negative review response strategy
        "pk_07091",  # Complaint Acknowledgment Response
    ],
    "write a SQL query to analyze database data": [
        "pk_19178",  # SQL Customer Segmentation
        "pk_18782",  # SQL Query for Monthly Revenue
        "pk_00054",  # Design E-commerce Database
        "pk_00632",  # Optimize slow database queries
        "pk_17577",  # Optimize Slow Database Queries
    ],
}


def precision_at_k(results, relevant_ids, k):
    top_k_ids = [r["id"] for r in results[:k]]
    hits = sum(1 for pid in top_k_ids if pid in relevant_ids)
    return hits / k


def reciprocal_rank(results, relevant_ids):
    for i, r in enumerate(results, start=1):
        if r["id"] in relevant_ids:
            return 1.0 / i
    return 0.0
def NDCG (results, relevant_ids, k):
    def dcg_at_k(results, relevant_ids, k):
        dcg = 0.0
        for i in range(k):
            if results[i]["id"] in relevant_ids:
                dcg += 1 / np.log2(i + 2)  # i+2 because rank starts at 1
        return dcg
    dcg = dcg_at_k(results, relevant_ids, k)
    def idcg_at_k(relevant_ids, k):
        ideal_dcg = sum(1 / np.log2(i + 2) for i in range(min(len(relevant_ids), k)))
        return ideal_dcg
    idcg = idcg_at_k(relevant_ids, k)
    return dcg / idcg if idcg > 0 else 0.0

def evaluate(collection, ground_truth, embedding_model, top_k=10, label="", query_prefix=""):
    p_scores, rr_scores, ndcg_scores = [], [], []
    for query, relevant_ids in ground_truth.items():
        if not relevant_ids:
            continue
        results, _ = retrieve_chromadb(query, collection, embedding_model, top_k=top_k, query_prefix=query_prefix)
        p = precision_at_k(results, set(relevant_ids), k=top_k)
        rr = reciprocal_rank(results, set(relevant_ids))
        ndcg = NDCG(results, set(relevant_ids), k=top_k)
        p_scores.append(p)
        rr_scores.append(rr)
        ndcg_scores.append(ndcg)
        print(f"  Query: '{query[:50]}' | P@{top_k}={p:.2f} | RR={rr:.2f} | NDCG@{top_k}={ndcg:.2f}")

    if p_scores:
        print(f"\n  Config {label} — Mean P@{top_k}: {np.mean(p_scores):.3f} | MRR: {np.mean(rr_scores):.3f} | Mean NDCG@{top_k}: {np.mean(ndcg_scores):.3f}")
    else:
        print(f"  Config {label} — No ground truth yet. Add relevant IDs to GROUND_TRUTH above.")


print("=== Config A (BGE) ===")
evaluate(config_a_chromadb, GROUND_TRUTH, bge_model, top_k=10, label="A")

print("\n=== Config B (E5) ===")
evaluate(config_b_chromadb, GROUND_TRUTH, e5_model, top_k=10, label="B", query_prefix="query: ")

=== Config A (BGE) ===
  Query: 'create a social media marketing campaign' | P@10=0.10 | RR=0.20 | NDCG@10=0.13
  Query: 'write a response to a negative customer review' | P@10=0.30 | RR=0.33 | NDCG@10=0.41
  Query: 'write a SQL query to analyze database data' | P@10=0.00 | RR=0.00 | NDCG@10=0.00

  Config A — Mean P@10: 0.133 | MRR: 0.178 | Mean NDCG@10: 0.182

=== Config B (E5) ===
  Query: 'create a social media marketing campaign' | P@10=0.10 | RR=0.33 | NDCG@10=0.17
  Query: 'write a response to a negative customer review' | P@10=0.20 | RR=1.00 | NDCG@10=0.46
  Query: 'write a SQL query to analyze database data' | P@10=0.00 | RR=0.00 | NDCG@10=0.00

  Config B — Mean P@10: 0.100 | MRR: 0.444 | Mean NDCG@10: 0.210


## 8. Export results for Step 4 (the reranker)

We package the top candidates for a given query into a clean format
that the reranker team (Step 4) can directly consume.
Each result includes `likes`, `upvotes`, and `difficulty` so the reranker can use them for metadata-aware scoring.

In [ ]:
import json

def get_candidates_for_reranker(query: str, top_k: int = 50):
    results_a, _ = retrieve_chromadb(query, config_a_chromadb, bge_model, top_k=top_k)
    results_b, _ = retrieve_chromadb(query, config_b_chromadb, e5_model, top_k=top_k, query_prefix="query: ")

    output = {
        "query": query,
        "top_k": top_k,
        "config_a_candidates": results_a,
        "config_b_candidates": results_b,
    }
    return output


# Hard-coded default query for reproducibility (Restart & Run All friendly).
# Change this string to test a different query.
QUERY = "help me write a professional email to a client"

candidates = get_candidates_for_reranker(QUERY, top_k=50)

with open("/content/candidates_for_reranker.json", "w") as f:
    json.dump(candidates, f, indent=2)

print(f"Query: {QUERY}")
print(f"Exported {len(candidates['config_a_candidates'])} candidates (Config A)")
print(f"Exported {len(candidates['config_b_candidates'])} candidates (Config B)")
print("Saved to: /content/candidates_for_reranker.json")

Query: help me write a professional email to a client
Exported 50 candidates (Config A)
Exported 50 candidates (Config B)
Saved to: /content/candidates_for_reranker.json


## RERANKER (PART 4)

Before building the reranker, let's do a sanity check

In [ ]:
print(config_a_bge.shape)
print(config_b_e5.shape)
print(metadata.shape)

(20000, 768)
(20000, 768)
(20000, 12)


In [ ]:
print("Chroma A:", config_a_chromadb.count())
print("Chroma B:", config_b_chromadb.count())

Chroma A: 20000
Chroma B: 20000


In [ ]:
candidates.keys()

dict_keys(['query', 'top_k', 'config_a_candidates', 'config_b_candidates'])

In [ ]:
#print example candidate
print("\nExample candidate from Config A:")
print(candidates["config_a_candidates"][49])
print("\nExample candidate from Config B:")
print(candidates["config_b_candidates"][49])



Example candidate from Config A:
{'rank': 50, 'id': 'pk_02240', 'similarity': 0.4395, 'distance': 0.5605, 'title': 'thank you email template', 'category': 'marketing', 'subcategory': 'email-marketing', 'tags': "['email-template', 'thank-you', 'review-request', 'ecommerce']", 'difficulty': 'beginner', 'likes': 419, 'upvotes': 288, 'content': 'hi i need a thank you email template for my online store basically after someone buys something i want to send them a thank you email and maybe ask for a review which i heard is important for social proof or whatever so if you could write me like a template that would be great that i can just copy and paste for each order also i want it to sound friendly and not too corporate or salesy because im a small handmade business and i want people to feel like theyre buying from a real person not some huge company also should i send it right away or wait a few days also i want to include some kind of discount code for their next purchase to get them to co

After performing vector retrieval we need to fine our search by performing a reranking over the best results that we got so that they can be ordered over certain criterias

In [ ]:

#as a model we will choose cross-encoder/ms-marco-MiniLM-L-6-v2, which is fine-tuned for relevance ranking tasks and should perform well in distinguishing subtle differences in prompt relevance.
from sentence_transformers import SentenceTransformer, util, CrossEncoder
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker_model = CrossEncoder(RERANKER_MODEL_NAME)
print(f"Reranker model loaded: {RERANKER_MODEL_NAME}")



Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker model loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In order to improve the reranker and make sure that it might perform also when the prompt is vague or unclear we want to incorporate in a structured textual representation also other attributes, like title, category and tags, so that it would improve our retrieval. This ensures also a coherence with "Config B" where said attributes where already taken into account for embedding.


Now time for the reranker function, that takes the "rough" candidates resulting from the vector search and reorders them using a stronger model, returning only the top k results.


In [ ]:


def rerank_candidates(query, candidates, top_k=10):
    query = query.strip() #remove leading/trailing whitespace
    candidate_texts = [candidate.get("content") for candidate in candidates] #retrieve the text representation for each candidate
    pairs = [[query, ct] for ct in candidate_texts] #prepare the input pairs for the cross-encoder (query + candidate text)
    scores = reranker_model.predict(pairs) #cross-encoder returns relevance scores for each pair (higher = more relevant)
    reranked = [{"candidate": c, "rerank_score": float(s)} for c, s in zip(candidates, scores)] #attach scores to candidates
    return sorted(reranked, key=lambda x: x["rerank_score"], reverse=True)[:top_k] #sort candidates by rerank_score and return top_k


Testing on candidates from Config A:

In [ ]:

reranked_a = rerank_candidates(QUERY, candidates["config_a_candidates"], top_k=5)

print("\nTop 5 reranked candidates from Config A:")
for i, item in enumerate(reranked_a, start=1):
    c = item["candidate"]
    score = item["rerank_score"]
    print(f"\n {i}  ID: {c['id']}  Rerank Score: {score:.4f}")
    print(f" Title: {c['title']}")
    print(f"content:{c['content']}")


Top 5 reranked candidates from Config A:

 1  ID: pk_07901  Rerank Score: 5.6294
 Title: Draft client email response
content:Act as a professional email writer. Draft a response to the following client inquiry about a delayed project delivery. The tone should be apologetic but confident, acknowledge their frustration, explain the reason for delay (resource constraints), outline the revised timeline with specific dates, and offer a 15% discount on the next phase as compensation. Keep the email under 200 words and end with a clear call to action to schedule a call to discuss further. Output the email in plain text format ready to send.

 2  ID: pk_01223  Rerank Score: 4.7215
 Title: Email Template for {{purpose}}
content:Write a professional email template for {{purpose}} communication. The tone should be {{tone}}, and it should be suitable for {{audience}}. Include subject line suggestions, opening greeting variations, body structure, and closing options. Make it adaptable by highlight

Now on Candidates from B

In [ ]:
# And now on the candidates from Config B
reranked_b = rerank_candidates(QUERY, candidates["config_b_candidates"], top_k=5)


print("\nTop 5 reranked candidates from Config B:")
for i, item in enumerate(reranked_b, start=1):
    c = item["candidate"]
    score = item["rerank_score"]
    print(f"\n {i}  ID: {c['id']}  Rerank Score: {score:.4f}")
    print(f" Title: {c['title']}")
    print(f"content:{c['content']}")



Top 5 reranked candidates from Config B:

 1  ID: pk_07901  Rerank Score: 5.6294
 Title: Draft client email response
content:Act as a professional email writer. Draft a response to the following client inquiry about a delayed project delivery. The tone should be apologetic but confident, acknowledge their frustration, explain the reason for delay (resource constraints), outline the revised timeline with specific dates, and offer a 15% discount on the next phase as compensation. Keep the email under 200 words and end with a clear call to action to schedule a call to discuss further. Output the email in plain text format ready to send.

 2  ID: pk_16889  Rerank Score: 5.0573
 Title: Draft ticket response for printer issue
content:Write a professional email response for a customer whose order arrived damaged. Acknowledge their frustration, apologize for the inconvenience, and explain the replacement process. Keep the tone empathetic but efficient.

 3  ID: pk_00072  Rerank Score: 4.9280


Now we want to analyze the reranker performance compared to the vector search for the top 10 among the 50 best ones found by vector search itself. As evaluation metrics we use NDCG, RR and P@10.

In [ ]:
def metric_calculator(results, relevant_ids, k):
    p = precision_at_k(results, set(relevant_ids), k=k)
    rr = reciprocal_rank(results, set(relevant_ids))
    ndcg = NDCG(results, set(relevant_ids), k=k)
    return p, rr, ndcg




def evaluate_vector_vs_reranker(ground_truth, candidate_k=50, top_k=10):
    rows = []
    for query, relevant_ids in ground_truth.items():
        if not relevant_ids:
            continue
        relevant_ids_set = set(str(x) for x in relevant_ids)
        res_a, _ = retrieve_chromadb(query,config_a_chromadb,bge_model,top_k=candidate_k)
        res_b, _ = retrieve_chromadb(query,config_b_chromadb,e5_model,top_k=candidate_k,query_prefix="query: ")
        vector_a_results = res_a[:top_k]
        vector_b_results = res_b[:top_k]
        reranked_a = rerank_candidates(query,res_a,top_k=top_k)
        reranked_b = rerank_candidates(query,res_b,top_k=top_k)
        reranked_a_results = [item["candidate"] for item in reranked_a]
        reranked_b_results = [item["candidate"] for item in reranked_b]
        row = {"query": query}
        systems = {"vector_a": vector_a_results,"vector_b": vector_b_results,"rerank_a": reranked_a_results,"rerank_b": reranked_b_results}
        for system_name, results in systems.items():
            metrics = metric_calculator(results,relevant_ids_set,top_k)
            for metric_name, value in zip(["precision", "reciprocal_rank", "ndcg"], metrics):
                row[f"{system_name}_{metric_name}"] = value
        rows.append(row)
    return pd.DataFrame(rows)


reranker_eval = evaluate_vector_vs_reranker(
    GROUND_TRUTH,
    candidate_k=50,
    top_k=10
)

print(reranker_eval)

                                            query  vector_a_precision  \
0        create a social media marketing campaign                 0.1   
1  write a response to a negative customer review                 0.3   
2      write a SQL query to analyze database data                 0.0   

   vector_a_reciprocal_rank  vector_a_ndcg  vector_b_precision  \
0                  0.200000       0.131205                 0.1   
1                  0.333333       0.413839                 0.2   
2                  0.000000       0.000000                 0.0   

   vector_b_reciprocal_rank  vector_b_ndcg  rerank_a_precision  \
0                  0.333333       0.169580                 0.0   
1                  1.000000       0.459972                 0.2   
2                  0.000000       0.000000                 0.0   

   rerank_a_reciprocal_rank  rerank_a_ndcg  rerank_b_precision  \
0                       0.0       0.000000                 0.0   
1                       1.0       0.553146   

In [ ]:

summary = pd.DataFrame({
    "method": ["Config A - Vector Search",
        "Config B - Vector Search",
        "Config A - Reranked",
        "Config B - Reranked"],
    "mean_p@10": [reranker_eval["vector_a_precision"].mean(),
        reranker_eval["vector_b_precision"].mean(),
        reranker_eval["rerank_a_precision"].mean(),
        reranker_eval["rerank_b_precision"].mean(),],
    "mrr": [reranker_eval["vector_a_reciprocal_rank"].mean(),
        reranker_eval["vector_b_reciprocal_rank"].mean(),
        reranker_eval["rerank_a_reciprocal_rank"].mean(),
        reranker_eval["rerank_b_reciprocal_rank"].mean(),],
    "mean_ndcg@10": [reranker_eval["vector_a_ndcg"].mean(),
        reranker_eval["vector_b_ndcg"].mean(),
        reranker_eval["rerank_a_ndcg"].mean(),
        reranker_eval["rerank_b_ndcg"].mean(),],
})
print(summary)


                     method  mean_p@10       mrr  mean_ndcg@10
0  Config A - Vector Search   0.133333  0.177778      0.181681
1  Config B - Vector Search   0.100000  0.444444      0.209851
2       Config A - Reranked   0.066667  0.333333      0.184382
3       Config B - Reranked   0.133333  0.333333      0.281179


FINAL STEP: METADATAS

## Metadata-Aware Reranking: Weighted Linear Combination

To incorporate metadata into the ranking process, we apply a weighted linear combination of semantic relevance and engagement signals. The semantic relevance score comes from the cross-encoder reranker, while likes and upvotes are used as metadata-based popularity signals.

Because these values live on different scales, each score is normalized before combination. The reranker score remains the dominant component, while metadata provides a smaller adjustment to promote prompts that are both relevant and positively received by users.


In [ ]:
reranked= rerank_candidates(QUERY, candidates["config_b_candidates"], top_k=50)
print(reranked[0])

{'candidate': {'rank': 3, 'id': 'pk_07901', 'similarity': 0.7356, 'distance': 0.2644, 'title': 'Draft client email response', 'category': 'productivity', 'subcategory': 'email-management', 'tags': "['email-writing', 'client-communication', 'professional-tone', 'apology-email', 'customer-relations']", 'difficulty': 'beginner', 'likes': 373, 'upvotes': 269, 'content': 'Act as a professional email writer. Draft a response to the following client inquiry about a delayed project delivery. The tone should be apologetic but confident, acknowledge their frustration, explain the reason for delay (resource constraints), outline the revised timeline with specific dates, and offer a 15% discount on the next phase as compensation. Keep the email under 200 words and end with a clear call to action to schedule a call to discuss further. Output the email in plain text format ready to send.'}, 'rerank_score': 5.629380226135254}


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def minmax_normalize(series):
    min_val = series.min()
    max_val = series.max()

    if max_val == min_val:
        return series.apply(lambda x: 0.0)

    return (series - min_val) / (max_val - min_val)
def build_metadata_text(candidate):
    return " ".join([str(candidate.get("title", "")),
        str(candidate.get("category", "")),
        str(candidate.get("subcategory",  "")),
        str(candidate.get("tags", "")),
        str(candidate.get("difficulty", ""))])
def weighted_reranker(query, candidates, metadata_weight=0.4, content_weight=0.6):
    query = query.strip()
    content_pairs= [[query, str(c.get("content", ""))] for c in candidates]
    metadata_pairs = [[query, build_metadata_text(c)] for c in candidates]
    content_scores = minmax_normalize(reranker_model.predict(content_pairs))
    metadata_scores = minmax_normalize(reranker_model.predict(metadata_pairs))
    combined_scores = metadata_weight * metadata_scores + content_weight * content_scores
    weighted_reranker_scores=[]
    for candidate, combined_score in zip(candidates, combined_scores):
        weighted_reranker_scores.append({"candidate": candidate, "weighted_rerank_score": float(combined_score)})
    return sorted(weighted_reranker_scores, key=lambda x: x["weighted_rerank_score"], reverse=True)

tfidf_vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
corpus_tfidf = tfidf_vectorizer.fit_transform(metadata["embedding_text"].astype(str).tolist())
corpus_texts = metadata["embedding_text"].astype(str).tolist()



def tf_idf_keyword_identifier(query, candidates, top_n=10):
    candidate_texts = [" ".join([build_metadata_text(c),str(c.get("content", ""))])for c in candidates]

    tfidf_matrix = tfidf_vectorizer.transform(candidate_texts)
    query_vector = tfidf_vectorizer.transform([query])

    cosine_similarities = cosine_similarity(query_vector,tfidf_matrix).flatten()

    feature_names = tfidf_vectorizer.get_feature_names_out()

    candidate_keywords = []
    keyword_details = []
    for i, candidate in enumerate(candidates):
        row = tfidf_matrix[i].toarray().flatten()
        top_indices = row.argsort()[::-1][:top_n]

        keywords = [feature_names[idx] for idx in top_indices if row[idx] > 0]

        candidate_keywords.append({"id": candidate.get("id", ""), "score": float(cosine_similarities[i]), "keywords": keywords})
        keyword_details.append({"id": candidate.get("id", ""), "title": candidate.get("title", ""), "tfidf_score": float(cosine_similarities[i]), "matched_keywords": keywords})
    return cosine_similarities.tolist(), candidate_keywords


def metadata_aware_reranker(query, candidates,top_k=10, alpha=0.75, beta=0.10, gamma=0.15):
    weighted_reranker_results=weighted_reranker(query, candidates, metadata_weight=0.4, content_weight=0.6)
    reranked_scores=pd.Series([candidate["weighted_rerank_score"] for candidate in weighted_reranker_results])
    popularity_scores=pd.Series([(float(candidate["candidate"].get("likes", 0))) + (float(candidate["candidate"].get("upvotes", 0))) for candidate in weighted_reranker_results])
    weighted_candidates=[item["candidate"] for item in weighted_reranker_results]
    keyword_scores, keywords=tf_idf_keyword_identifier(query, weighted_candidates, top_n=10)

    final_scores=alpha*minmax_normalize(reranked_scores) + beta*minmax_normalize(popularity_scores) + gamma*minmax_normalize(pd.Series(keyword_scores))
    for i, item in enumerate(weighted_reranker_results):
        item["final_score"] = float(final_scores.iloc[i])
        item["tfidf_keywords"] = keywords[i]["keywords"]

    return sorted(weighted_reranker_results, key=lambda x: x["final_score"], reverse=True)[:top_k]

In [ ]:
print("\nTop 5 candidates from Config A with metadata-aware reranking:")

final_reranked_a = metadata_aware_reranker( QUERY, candidates["config_a_candidates"], top_k=5, alpha=0.75, beta=0.10, gamma=0.15)

for i, item in enumerate(final_reranked_a, start=1):
    c = item["candidate"]
    score = item["final_score"]

    print(f"\n{i}  ID: {c['id']}  Final Score: {score:.4f}")
    print(f"Title: {c['title']}")
    print(f"Content: {c['content']}")
    print(f"TF-IDF keywords: {item['tfidf_keywords']}")
    print(f"number of likes: {c['likes']} ")
    print(f"number of upvotes: {c['upvotes']}")
    print(f"Dataset tags: {c['tags']}")


Top 5 candidates from Config A with metadata-aware reranking:

1  ID: pk_07901  Final Score: 0.9908
Title: Draft client email response
Content: Act as a professional email writer. Draft a response to the following client inquiry about a delayed project delivery. The tone should be apologetic but confident, acknowledge their frustration, explain the reason for delay (resource constraints), outline the revised timeline with specific dates, and offer a 15% discount on the next phase as compensation. Keep the email under 200 words and end with a clear call to action to schedule a call to discuss further. Output the email in plain text format ready to send.
TF-IDF keywords: ['email', 'client', 'the email', 'draft', 'call', 'tone', 'professional', 'response', 'the', 'as']
number of likes: 373 
number of upvotes: 269
Dataset tags: ['email-writing', 'client-communication', 'professional-tone', 'apology-email', 'customer-relations']

2  ID: pk_12525  Final Score: 0.8164
Title: Write profession

In [ ]:
print("\nTop 5 candidates from Config B with metadata-aware reranking:")

final_reranked_b = metadata_aware_reranker( QUERY, candidates["config_b_candidates"], top_k=5, alpha=0.75, beta=0.10, gamma=0.15)

for i, item in enumerate(final_reranked_b, start=1):
    c = item["candidate"]
    score = item["final_score"]

    print(f"\n{i}  ID: {c['id']}  Final Score: {score:.4f}")
    print(f"Title: {c['title']}")
    print(f"Content: {c['content']}")
    print(f"TF-IDF keywords: {item['tfidf_keywords']}")
    print(f"number of likes: {c['likes']} ")
    print(f"number of upvotes: {c['upvotes']}")
    print(f"Dataset tags: {c['tags']}")


Top 5 candidates from Config B with metadata-aware reranking:

1  ID: pk_07901  Final Score: 0.9908
Title: Draft client email response
Content: Act as a professional email writer. Draft a response to the following client inquiry about a delayed project delivery. The tone should be apologetic but confident, acknowledge their frustration, explain the reason for delay (resource constraints), outline the revised timeline with specific dates, and offer a 15% discount on the next phase as compensation. Keep the email under 200 words and end with a clear call to action to schedule a call to discuss further. Output the email in plain text format ready to send.
TF-IDF keywords: ['email', 'client', 'the email', 'draft', 'call', 'tone', 'professional', 'response', 'the', 'as']
number of likes: 373 
number of upvotes: 269
Dataset tags: ['email-writing', 'client-communication', 'professional-tone', 'apology-email', 'customer-relations']

2  ID: pk_12525  Final Score: 0.7911
Title: Write profession

In [ ]:
def evaluate_vector_reranker_metadata(ground_truth, candidate_k=50, top_k=10):
    rows = []
    for query, relevant_ids in ground_truth.items():
        if not relevant_ids:
            continue
        relevant_ids_set = set(relevant_ids)
        res_a, _ = retrieve_chromadb(query, config_a_chromadb, bge_model, top_k=candidate_k)
        res_b, _ = retrieve_chromadb(query, config_b_chromadb, e5_model, top_k=candidate_k, query_prefix="query: ")
        #metadata-aware reranked top k
        metadata_reranked_a = metadata_aware_reranker(query, res_a, top_k=top_k)
        metadata_reranked_b = metadata_aware_reranker(query, res_b,  top_k=top_k)
        metadata_reranked_a_results = [item["candidate"] for item in metadata_reranked_a]
        metadata_reranked_b_results = [item["candidate"] for item in metadata_reranked_b]
        #calculate metrics
        p_a_metadata = precision_at_k(metadata_reranked_a_results, relevant_ids_set, k=top_k)
        rr_a_metadata = reciprocal_rank(metadata_reranked_a_results, relevant_ids_set)
        NDCGk_a_metadata = NDCG(metadata_reranked_a_results, relevant_ids_set, k=top_k)

        p_b_metadata = precision_at_k(metadata_reranked_b_results, relevant_ids_set, k=top_k)
        rr_b_metadata = reciprocal_rank(metadata_reranked_b_results, relevant_ids_set)
        NDCGk_b_metadata = NDCG(metadata_reranked_b_results, relevant_ids_set, k=top_k)

        rows.append({"query": query,
            "metadata_rerank_a_p@10": precision_at_k(metadata_reranked_a_results, relevant_ids_set, k=top_k),
            "metadata_rerank_a_rr": rr_a_metadata,
            "metadata_rerank_a_ndcg@10": NDCGk_a_metadata,
            "metadata_rerank_b_p@10": precision_at_k(metadata_reranked_b_results, relevant_ids_set, k=top_k),
            "metadata_rerank_b_rr": rr_b_metadata,
            "metadata_rerank_b_ndcg@10": NDCGk_b_metadata,})

    return pd.DataFrame(rows)

In [ ]:
metadata_eval = evaluate_vector_reranker_metadata(GROUND_TRUTH, candidate_k=50, top_k=10)

print(metadata_eval[["query","metadata_rerank_a_p@10", "metadata_rerank_a_rr", "metadata_rerank_a_ndcg@10","metadata_rerank_b_p@10", "metadata_rerank_b_rr", "metadata_rerank_b_ndcg@10"]])

full_eval= reranker_eval.merge(
    metadata_eval,
    on="query",
    how="inner"
)
print(full_eval[["query", "vector_a_precision", "vector_a_reciprocal_rank", "vector_a_ndcg",
    "vector_b_precision", "vector_b_reciprocal_rank", "vector_b_ndcg",
    "rerank_a_precision", "rerank_a_reciprocal_rank", "rerank_a_ndcg",
    "rerank_b_precision", "rerank_b_reciprocal_rank", "rerank_b_ndcg",
    "metadata_rerank_a_p@10", "metadata_rerank_a_rr", "metadata_rerank_a_ndcg@10",
    "metadata_rerank_b_p@10", "metadata_rerank_b_rr", "metadata_rerank_b_ndcg@10"
]])


full_summary = pd.DataFrame({
    "method": ["Config A - Vector Search",
        "Config B - Vector Search",
        "Config A - Reranked",
        "Config B - Reranked",
        "Config A - Metadata-aware Rerank",
        "Config B - Metadata-aware Rerank"],
    "mean_p@10": [full_eval["vector_a_precision"].mean(),
        full_eval["vector_b_precision"].mean(),
        full_eval["rerank_a_precision"].mean(),
        full_eval["rerank_b_precision"].mean(),
        full_eval["metadata_rerank_a_p@10"].mean(),
        full_eval["metadata_rerank_b_p@10"].mean(),],
    "mrr": [full_eval["vector_a_reciprocal_rank"].mean(),
        full_eval["vector_b_reciprocal_rank"].mean(),
        full_eval["rerank_a_reciprocal_rank"].mean(),
        full_eval["rerank_b_reciprocal_rank"].mean(),
        full_eval["metadata_rerank_a_rr"].mean(),
        full_eval["metadata_rerank_b_rr"].mean(),],
    "mean_ndcg@10": [full_eval["vector_a_ndcg"].mean(),
        full_eval["vector_b_ndcg"].mean(),
        full_eval["rerank_a_ndcg"].mean(),
        full_eval["rerank_b_ndcg"].mean(),
        full_eval["metadata_rerank_a_ndcg@10"].mean(),
        full_eval["metadata_rerank_b_ndcg@10"].mean(),]})
print("\nFull Retrieval/Reranking Summary:")
print(full_summary)



                                            query  metadata_rerank_a_p@10  \
0        create a social media marketing campaign                     0.0   
1  write a response to a negative customer review                     0.3   
2      write a SQL query to analyze database data                     0.0   

   metadata_rerank_a_rr  metadata_rerank_a_ndcg@10  metadata_rerank_b_p@10  \
0                   0.0                   0.000000                     0.0   
1                   0.5                   0.504378                     0.5   
2                   0.0                   0.000000                     0.0   

   metadata_rerank_b_rr  metadata_rerank_b_ndcg@10  
0                   0.0                   0.000000  
1                   0.5                   0.757439  
2                   0.0                   0.000000  
                                            query  vector_a_precision  \
0        create a social media marketing campaign                 0.1   
1  write a response 